In [1]:
import mlrun
# Loads AWS_ACCESS_KEY_ID and AWS_SECRET_ACCESS_KEY and MLRUN_AWS_ROLE_ARN
from dotenv import load_dotenv
load_dotenv() 

from pathlib import Path
from datetime import datetime

artifact_path = Path.cwd().parent
artifact_path = str(artifact_path.as_posix()) # convert windows path to unix path
artifact_path = "file://" + artifact_path
p = mlrun.set_environment("http://localhost:8080", artifact_path=artifact_path)

project = mlrun.load_project(name='finetune-legal-extractor', context="../") # project yaml must be in this directory
# Verify it loaded correctly by checking its status or printing the config
# print(project.to_yaml())

> 2026-06-08 13:17:40,259 [warning] Client version with higher version than server version isn't supported, align your client to the server version: {"parsed_client_version":"Version(major=1, minor=11, patch=0, prerelease=None, build=None)","parsed_server_version":"Version(major=1, minor=10, patch=3, prerelease=None, build=None)"}
> 2026-06-08 13:17:40,264 [warning] Client version with higher version than server version isn't supported, align your client to the server version: {"parsed_client_version":"Version(major=1, minor=11, patch=0, prerelease=None, build=None)","parsed_server_version":"Version(major=1, minor=10, patch=3, prerelease=None, build=None)"}


In [2]:
import mlrun
from mlrun.model import RunObject

UID = "97082ffd06f34c8d9807a631585f2250"

# Initialize the MLRun DB client
db = mlrun.get_run_db()
run_dict = db.read_run(uid=UID, project="finetune-legal-extractor")
# Convert the dictionary to a RunObject for easier API access
run = RunObject.from_dict(run_dict)

# Get run outputs
run_parameters = run_dict['spec']['parameters']
run_metrics = run_dict['status']['results']
output = run.outputs['return'] # this is what was returned 

print(run_parameters)
print(run_metrics)
print(output)

{'train_dataset': 'raw-proc-process-raw_train_data', 'train_dataset_tag': '20260506_1224', 'val_dataset': 'raw-proc-process-raw_validation_data', 'val_dataset_tag': '20260506_1224', 'test_dataset': 'raw-proc-process-raw_test_data', 'test_dataset_tag': '20260506_1224', 'prompt': 'contract_extractor_prompt', 'prompt_tag': '20260603_1957', 'epochs': 5, 'batch_grad_accumulation': 16, 'learning_rate': 0.0002, 'lora_r': 16, 'lora_alpha': 32, 'early_stopping_threshold': 0.001}
{'count': 121, 'average_accuracy': 0.8784148326577259, 'average_fmeasure': 0.820151897229756, 't_average_fmeasure': 0.9165791720415065, 't_average_perc_above_75fmeasure': 0.8551596898706432, 'f_average_fmeasure': 0.11773311345873137, 'f_average_perc_above_75fmeasure': 0.09944903581267217, 'min_accuracy': 0.7058823529411765, 'min_t_average_fmeasure': 0.7105234783986872, 'min_t_perc_above_75fmeasure': 0.4666666666666667, 'min_f_average_fmeasure': 0.0, 'min_f_perc_above_75fmeasure': 0.0, 'return': {'commit_oid:': '39c89f59

In [5]:
# Pass in model_id, commit, hyperparameters, performance metrics

version = datetime.now().strftime("%Y%m%d_%H%M")
hyperparams = run_parameters

project.log_model(key="Hermes-4-14B-legal-extractor-adapter",
                  tag=version,
                  metrics=run_metrics,
                  parameters=run_parameters,
                  framework="Hugging Face model",
                  model_url="https://huggingface.co/JerroldK/H4-14b-contract-extractor-adapter",
                  labels={"model": "Hermes-4-14B"},
                  upload=False
)

In [8]:
model = project.get_artifact(key="Hermes-4-14B-legal-extractor-adapter",
                             tag='latest')
print(model.tag)
print(model.labels)
print(model.model_url)
print(model.metrics)
print(model.parameters)

latest
{'model': 'Hermes-4-14B', 'framework': 'Hugging Face model'}
https://huggingface.co/JerroldK/H4-14b-contract-extractor-adapter
{'count': 121, 'average_accuracy': 0.8784148326577259, 'average_fmeasure': 0.820151897229756, 't_average_fmeasure': 0.9165791720415065, 't_average_perc_above_75fmeasure': 0.8551596898706432, 'f_average_fmeasure': 0.11773311345873137, 'f_average_perc_above_75fmeasure': 0.09944903581267217, 'min_accuracy': 0.7058823529411765, 'min_t_average_fmeasure': 0.7105234783986872, 'min_t_perc_above_75fmeasure': 0.4666666666666667, 'min_f_average_fmeasure': 0.0, 'min_f_perc_above_75fmeasure': 0.0, 'return': {'commit_oid:': '39c89f599964a53e6dc2e11c273a6d2d6ad52a2e', 's3_output_path': 's3://legal-llama-data/training/20260603_1558/evaluation/metrics.json'}}
{'train_dataset': 'raw-proc-process-raw_train_data', 'train_dataset_tag': '20260506_1224', 'val_dataset': 'raw-proc-process-raw_validation_data', 'val_dataset_tag': '20260506_1224', 'test_dataset': 'raw-proc-process

In [ ]:
# from mlrun.artifacts import ModelArtifact

# # Create a remote model artifact 
# remote_model = ModelArtifact(
#     model_url="https://huggingface.co/JerroldK/Hermes-4-14B-contract-extractor"
# )

# # You can attach the commit hash or revision to the artifact metadata
# remote_model.labels = {"commit_hash": "75875f970c359f89ad9e7d4dc86bf3c075c73c31"}

# # Log it to the project
# project.log_artifact(remote_model, key="Hermes-4-14B-contract-extractor")

In [7]:
project.save(store=True)